In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split, TensorDataset

import shap 

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tqdm import tqdm

import sys
sys.path.append('../src')

from preprocessing import *
from models import  *

from autogluon.tabular import TabularPredictor

from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [2]:
dfs = get_dfs(os.path.dirname(os.getcwd()))
static_df = create_static_df(dfs)
medication_df = create_medication_df(dfs)
vitals_ca, vitals_lab = create_vitals_df(dfs)

ts_data = create_ts_data(vitals_ca, vitals_lab, medication_df, merge_lab=True, merge_med=True, static_df=static_df)
#forward_fill_imputation(ts_data) 

#notes = create_notes_df(dfs, filename='../data/embeddings/emb_gte.npy')
notes = create_notes_df(dfs, filename='../data/embeddings/emb_med_gte_simcse_en_ger.npy')
#notes = create_notes_df(dfs, filename=None)

biopsy_df = dfs['biopsy']
eligible_patient_ids = get_valid_patient_ids(static_df=static_df, ts_data=ts_data, notes_df=notes)
datapoints_limit = len(eligible_patient_ids)
#datapoints_limit = 320

dataset_splits = create_dataset_splits(
    static_df=static_df,
    ts_data=ts_data,
    notes_df=notes,
    biopsy_df=biopsy_df,
    train_size=0.8,
    test_size=0.2,
    max_patients=datapoints_limit,
    random_state=42,
)

full_dataset = dataset_splits['full']
train_dataset = dataset_splits['train']
test_dataset = dataset_splits['test']
ts_scaler = train_dataset.ts_scaler
static_scaler = train_dataset.scaler


Unique patients in medication: 3335
Removing patients that are not in static_df
Unique patients in clinical assessments: 3296
Average entries per patient 78.9
Unique patients in clinical assessments: 3465
Removing patients that are not in static_df
Unique patients in clinical assessments: 3423
Average entries per patient 61.16155419222904
Unique patients in lab df: 3460
Removing patients that are not in static_df
Unique patients in lab df: 3410
Average entries per patient 472.2140762463343
Reading notes from exams.csv
Found 206509 texts
Loading 215137 texts from clinical assessments
Concatenated texts and deleted NaNs, final count: 367140
Average texts per patient: 106.9


In [3]:
batch_size = 16
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

#vanilla_lstm = VanillaTimeSeriesEncoder()
att_encoder = TimeAwareAttentionEncoder(use_temporal_attention=True)
model = MultiModal(att_encoder, categorical_cardinalities=train_dataset.categorical_cardinalities, use_static=True, use_notes=True).to(device)
predict_steps_ahead = 1

model.load_state_dict(torch.load('../models/full10ep.pt', weights_only=True))


<All keys matched successfully>

In [5]:
def extract_horizon_reprs(
    dataloader, 
    model, 
    horizons, 
    label_key, 
    rel_days_key=None,
    min_history_days=90,
    max_days=180,
    max_samples_per_patient=5
):
    """
    - Feeds each patient's entire time series (minus last step) in one pass.
    - Extracts hidden representations for each time step from model output.
    - For each horizon H, determines if the event occurs within H days from that step.
    - Only includes time steps between min_history_days and max_days.
    - Takes up to max_samples_per_patient samples per patient (first N valid samples).

    If `label_key` corresponds to a multi-day event list (like "rej_rel_days"), then `rel_days_key`
    can be left None (ignored), and we will handle the logic differently.
    
    Parameters:
    -----------
    dataloader : DataLoader
        The PyTorch dataloader yielding patient data batches
    model : torch.nn.Module
        The trained model to extract representations from
    horizons : list
        List of horizon values (in days) to consider
    label_key : str
        Key in batch dictionary for labels
    rel_days_key : str, optional
        Key for relative days to event (only for single-event labels)
    min_history_days : int, default=90
        Minimum number of days of history required
    max_days : int, default=180
        Maximum number of days to include in the dataset
    max_samples_per_patient : int, default=5
        Maximum number of samples to include per patient
    """
    from tqdm import tqdm
    
    model.eval()

    # For each horizon, prepare storage for hidden reps, labels, day-of-step, patient_id
    hr_repr = {H: [] for H in horizons}
    hr_label = {H: [] for H in horizons}
    hr_days  = {H: [] for H in horizons}
    hr_pids  = {H: [] for H in horizons}
    
    # Keep track of samples per patient for each horizon
    patient_sample_counts = {H: {} for H in horizons}

    all_pids = set()
    positive_pids = set()

    with torch.no_grad():
        # Add progress bar for the dataloader iteration
        for batch in tqdm(dataloader, desc="Processing patients"):
            pid  = batch['patient_id']
            slen = batch['seq_len']

            # Static features
            cat_static = batch['static_categorical_features'].to(device)
            num_static = batch['static_numerical_features'].to(device)

            # Time series
            full_ts    = batch['ts_features'].to(device)  # shape (B, T, F)
            timesteps  = batch['timesteps'].to(device)    # shape (B, T)
            mask_      = batch['mask'].to(device)         # shape (B, T)
            value_mask_full = batch['value_mask'].to(device)  # shape (B, T, F)

            # --- Convert single-element Tensors in label_key to float/list ---
            raw_labels_data = batch[label_key]  # shape (B,)
            labels_data = []
            for val in raw_labels_data:
                if isinstance(val, torch.Tensor):
                    # If it's a single element, convert to float
                    if val.numel() == 1:
                        val = float(val.item())
                    else:
                        # If multi-element, convert to NumPy or list
                        val = val.cpu().numpy()
                # Otherwise, val can be float, int, list, etc.
                labels_data.append(val)

            # If single-event usage, we also have rel_days_key => shape (B,)
            if rel_days_key and rel_days_key in batch:
                raw_rel_days_data = batch[rel_days_key]
                rel_days_data = []
                for dval in raw_rel_days_data:
                    if isinstance(dval, torch.Tensor):
                        if dval.numel() == 1:
                            dval = float(dval.item())
                        else:
                            dval = dval.cpu().numpy()
                    rel_days_data.append(dval)
                rel_days_data = np.array(rel_days_data)
            else:
                rel_days_data = None  # We'll handle multi-day logic below

            # If using notes
            notes_embeddings = batch['notes_embeddings'].to(device)
            notes_timesteps  = batch['notes_timesteps'].to(device)
            notes_mask       = batch['notes_mask'].to(device)

            B, T, F = full_ts.shape

            # Loop over each patient in this batch
            for i in range(B):
                patient_id_i = pid[i]
                all_pids.add(patient_id_i)

                # label_or_list => single-event (float 0/1) or multi-event list
                label_or_list = labels_data[i]

                # Keep track of which patients have at least one event
                if isinstance(label_or_list, (int, float, np.number)):
                    # single label
                    if label_or_list == 1:
                        positive_pids.add(patient_id_i)
                elif isinstance(label_or_list, (list, np.ndarray)):
                    # multiple days => if not empty => event
                    if len(label_or_list) > 0:
                        positive_pids.add(patient_id_i)
                elif label_or_list is not None:
                    # Unknown type
                    raise TypeError(f"Unsupported label type: {type(label_or_list)}")

                # If the sequence is too short
                if slen[i] < 2:
                    continue
                
                # Check if we already have max samples for this patient for all horizons
                if max_samples_per_patient > 0:
                    all_horizons_at_max = True
                    for H in horizons:
                        if patient_id_i not in patient_sample_counts[H] or patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            all_horizons_at_max = False
                            break
                    
                    if all_horizons_at_max:
                        continue  # Skip this patient altogether if already at max for all horizons

                # Slice the valid portion of the time series: [0..slen[i]-1]
                seq_len_i = slen[i].item()
                ts_i = full_ts[i:i+1, :seq_len_i, :]   # (1, seq_len_i, F)
                tm_i = timesteps[i:i+1, :seq_len_i]    # (1, seq_len_i)
                mk_i = mask_[i:i+1, :seq_len_i]        # (1, seq_len_i)

                # Model input => omit last step from time series
                inp_seq = ts_i[:, :-1, :]              # (1, seq_len_i-1, F)
                elapsed_times = tm_i[:, 1:] - tm_i[:, :-1]
                inp_mask = mk_i[:, :-1]
                inp_value_mask = value_mask_full[i:i+1, :seq_len_i-1, :]

                notes_emb_i = notes_embeddings[i:i+1]
                notes_ts_i  = notes_timesteps[i:i+1]
                notes_mk_i  = notes_mask[i:i+1]

                # Forward pass
                out, lstm_out, _, _ = model(
                    x=inp_seq,
                    elapsed_times=elapsed_times,     # (1, seq_len_i - 1)
                    timesteps=tm_i[:, :-1],          # (1, seq_len_i - 1)
                    notes_embeddings=notes_emb_i,
                    notes_timesteps=notes_ts_i,
                    static_features=(cat_static[i:i+1], num_static[i:i+1]),
                    mask=inp_mask,                   # (1, seq_len_i - 1)
                    notes_mask=notes_mk_i,
                    value_mask=inp_value_mask
                )
                # lstm_out => shape (1, seq_len_i-1, hidden_size)

                # Time array for all steps
                time_arr = tm_i.cpu().numpy().flatten()  # (seq_len_i,)
                
                # Initialize counts for this patient if not present
                for H in horizons:
                    if patient_id_i not in patient_sample_counts[H]:
                        patient_sample_counts[H][patient_id_i] = 0
                
                # Loop through time steps and only process until we reach max_samples for each horizon
                for k in range(seq_len_i - 1):
                    cur_day = time_arr[k+1]  # day of the (k+1)-th step
                    
                    # Skip if outside the desired range
                    if cur_day < min_history_days or cur_day > max_days:
                        continue
                    
                    # Get representation for this time step
                    rep_ = lstm_out[0, k, :].cpu().numpy()
                    
                    # Check for each horizon if we still need more samples
                    any_horizon_needs_samples = False
                    for H in horizons:
                        if patient_sample_counts[H][patient_id_i] < max_samples_per_patient:
                            any_horizon_needs_samples = True
                            break
                    
                    if not any_horizon_needs_samples:
                        break  # Exit time step loop if all horizons have enough samples
                    
                    # Process for each horizon that still needs samples
                    for H in horizons:
                        # Skip if already have max samples for this horizon
                        if patient_sample_counts[H][patient_id_i] >= max_samples_per_patient:
                            continue
                        
                        # If single-event logic is in play
                        if rel_days_data is not None:
                            event_label = label_or_list     # 0 or 1
                            event_day   = rel_days_data[i]  # single day
                            if (event_label == 1) and (0 < (event_day - cur_day) <= H):
                                label_ = 1
                            else:
                                label_ = 0

                        # Else multi-event logic (like rejections)
                        else:
                            # label_or_list is a list of event days or None
                            if isinstance(label_or_list, (list, np.ndarray)) and len(label_or_list) > 0:
                                # label_ = 1 if any day d in label_or_list satisfies (0 < d - cur_day <= H)
                                label_ = int(any(0 < (d - cur_day) <= H for d in label_or_list))
                            else:
                                label_ = 0

                        # Store
                        hr_repr[H].append(rep_)
                        hr_label[H].append(label_)
                        hr_days[H].append(cur_day)
                        hr_pids[H].append(patient_id_i)
                        
                        # Update sample count
                        patient_sample_counts[H][patient_id_i] += 1

    # Convert lists to numpy arrays for convenience
    for H in horizons:
        hr_repr[H] = np.array(hr_repr[H])
        hr_label[H] = np.array(hr_label[H])
        hr_days[H]  = np.array(hr_days[H])
        hr_pids[H]  = np.array(hr_pids[H])

    # Calculate and print statistics
    for H in horizons:
        total_patients = len(patient_sample_counts[H])
        avg_samples = np.mean([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]])
        max_samples = max([patient_sample_counts[H][pid] for pid in patient_sample_counts[H]]) if patient_sample_counts[H] else 0
        
        print(f"Horizon {H}: {total_patients} patients, avg {avg_samples:.1f} samples/patient, max {max_samples} samples/patient")

    print(f"Total unique patients: {len(all_pids)}, patients with event: {len(positive_pids)}")
    return hr_repr, hr_label, hr_days, hr_pids

In [6]:
horizons = [30, 90, 180]
min_history_days = 90
max_days = 720

print("Embedding patients...")

train_graft_repr, train_graft_lbl, train_graft_days, train_graft_pids = extract_horizon_reprs(train_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)
test_graft_repr, test_graft_lbl, test_graft_days, test_graft_pids = extract_horizon_reprs(test_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)

train_rej_repr, train_rej_lbl, train_rej_days, train_rej_pids = extract_horizon_reprs(train_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)
test_rej_repr, test_rej_lbl, test_rej_days, test_rej_pids = extract_horizon_reprs(test_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)

train_mort_repr, train_mort_lbl, train_mort_days, train_mort_pids = extract_horizon_reprs(train_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)
test_mort_repr, test_mort_lbl, test_mort_days, test_mort_pids = extract_horizon_reprs(test_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)

# Print some stats about the extracted data
for H in horizons:
    print(f"\n===== Extracted Features for Horizon {H} days =====")
    print(f"Graft Loss Training: {train_graft_repr[H].shape}, Positive: {sum(train_graft_lbl[H])}, Ratio: {sum(train_graft_lbl[H])/len(train_graft_lbl[H]):.4f}")
    print(f"Graft Loss Testing: {test_graft_repr[H].shape}, Positive: {sum(test_graft_lbl[H])}, Ratio: {sum(test_graft_lbl[H])/len(test_graft_lbl[H]):.4f}")
    print(f"Rejection Training: {train_rej_repr[H].shape}, Positive: {sum(train_rej_lbl[H])}, Ratio: {sum(train_rej_lbl[H])/len(train_rej_lbl[H]):.4f}")
    print(f"Rejection Testing: {test_rej_repr[H].shape}, Positive: {sum(test_rej_lbl[H])}, Ratio: {sum(test_rej_lbl[H])/len(test_rej_lbl[H]):.4f}")

Embedding patients...


Processing patients: 100%|██████████| 170/170 [03:12<00:00,  1.14s/it]


Horizon 30: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 2706, patients with event: 461


Processing patients: 100%|██████████| 43/43 [00:45<00:00,  1.05s/it]


Horizon 30: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 676, patients with event: 125


...

Horizon 30: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 676, patients with event: 53


Processing patients: 100%|██████████| 170/170 [03:13<00:00,  1.14s/it]


Horizon 30: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 2706 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 2706, patients with event: 846


Processing patients: 100%|██████████| 43/43 [00:46<00:00,  1.07s/it]

Horizon 30: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 676 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 676, patients with event: 192

===== Extracted Features for Horizon 30 days =====
Graft Loss Training: (12969, 512), Positive: 26, Ratio: 0.0020
Graft Loss Testing: (3229, 512), Positive: 9, Ratio: 0.0028
Rejection Training: (12969, 512), Positive: 135, Ratio: 0.0104
Rejection Testing: (3229, 512), Positive: 23, Ratio: 0.0071

===== Extracted Features for Horizon 90 days =====
Graft Loss Training: (12969, 512), Positive: 70, Ratio: 0.0054
Graft Loss Testing: (3229, 512), Positive: 43, Ratio: 0.0133
Rejection Training: (12969, 512), Positive: 210, Ratio: 0.0162
Rejection Testing: (3229, 512), Positive: 48, Ratio: 0.0149

===== Extracted Features for Horizon 180 days =====
Graft Loss Training: (12969, 512), Positive: 122, Ratio: 0.0094
Graft Loss Te

In [7]:
### LOGISTIC REGRESSION

def train_and_eval_logistic(X_train, y_train, X_test, y_test, event_name="Event"):
    clf = LogisticRegression(class_weight={0:1, 1:10},max_iter=1000).fit(X_train, y_train)
    y_pred_proba = clf.predict_proba(X_test)[:, 1]
    
    threshold = .2
    y_pred = (y_pred_proba >= threshold).astype(int)

    #y_pred = clf.predict(X_test)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    print(f"\n{event_name} Prediction:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")

for H in horizons:
    train_and_eval_logistic(train_graft_repr[H], train_graft_lbl[H], test_graft_repr[H], test_graft_lbl[H], event_name=f"GraftLoss@{H}")
    #train_and_eval_logistic(train_mort_repr[H], train_mort_lbl[H], test_mort_repr[H], test_mort_lbl[H], event_name=f"Mortality@{H}")
    #train_and_eval_logistic(train_rej_repr[H], train_rej_lbl[H], test_rej_repr[H],  test_rej_lbl[H], event_name=f"Rejection@{H}")



GraftLoss@30 Prediction:
AUC = 0.7421
Acc = 0.9963
Prec = 0.0000
Recall (Sensitivity) = 0.0000
Specificity = 0.9991
F1 = 0.0000

GraftLoss@90 Prediction:
AUC = 0.7651
Acc = 0.9768
Prec = 0.0556
Recall (Sensitivity) = 0.0465
Specificity = 0.9893
F1 = 0.0506

GraftLoss@180 Prediction:
AUC = 0.7504
Acc = 0.9644
Prec = 0.0781
Recall (Sensitivity) = 0.0820
Specificity = 0.9814
F1 = 0.0800


In [8]:
def train_and_eval_mlp(
    X_train, y_train, 
    X_test, y_test, 
    event_name="Event", 
    epochs=50, 
    batch_size=32, 
    lr=5e-3, 
    use_upsampling=True,
    eval_interval=5
):
    # Convert data to tensors and move to device
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(device)

    # Create a DataLoader for training
    train_dataset = TensorDataset(X_train_t, y_train_t)
    train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Define the model on device
    model_mlp = SimpleMLP(input_dim=X_train.shape[1]).to(device)
    optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)

    # Define loss
    if use_upsampling:
        # Example: Increase the loss weight for the positive class
        pos_weight = torch.tensor([25.0], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    # Evaluation function
    def evaluate_model(epoch=None):
        """Compute metrics on the test set and print them."""
        model_mlp.eval()
        with torch.no_grad():
            logits_test = model_mlp(X_test_t)          # shape (N,)
            probs_test  = torch.sigmoid(logits_test)   # shape (N,)
            y_pred      = (probs_test >= 0.5).int().cpu().numpy()
            y_prob      = probs_test.cpu().numpy()
            y_true      = y_test_t.cpu().numpy()

        # Metrics
        auc_  = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')
        acc_  = accuracy_score(y_true, y_pred)
        prec_ = precision_score(y_true, y_pred, zero_division=0)
        rec_  = recall_score(y_true, y_pred, zero_division=0)
        f1_   = f1_score(y_true, y_pred, zero_division=0)
        
        # Specificity = TN / (TN + FP)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        # Print
        msg_prefix = f"[{event_name}]"
        if epoch is not None:
            msg_prefix += f" Epoch {epoch}/{epochs}"
        print(
            f"{msg_prefix}\n"
            f"AUC = {auc_:.4f}, Acc = {acc_:.4f}, "
            f"Prec = {prec_:.4f}, Recall = {rec_:.4f}, "
            f"Spec = {specificity:.4f}, F1 = {f1_:.4f}\n"
        )

        model_mlp.train()  # Switch back to training mode

    # Training loop
    model_mlp.train()
    for epoch in range(1, epochs + 1):
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model_mlp(batch_x)    # shape (B,)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}", end='\r')

        # Evaluate every N epochs (e.g., every 5 epochs)
        if epoch % eval_interval == 0 or epoch == epochs:
            # Move to a new line so the loss print isn't overwritten
            print()
            evaluate_model(epoch=epoch)

    # Final model save
    model_mlp.cpu()
    torch.save(model_mlp.state_dict(), f'../models/{event_name}_clf.pth')


# Example usage
for H in horizons:
    train_and_eval_mlp( X_train=train_graft_repr[H], y_train=train_graft_lbl[H], X_test=test_graft_repr[H], y_test=test_graft_lbl[H], # pos weight 25
                       event_name=f"GraftLoss@{H}", epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2)

    # Rejections
    train_and_eval_mlp(X_train=train_rej_repr[H],y_train=train_rej_lbl[H],X_test=test_rej_repr[H], y_test=test_rej_lbl[H],
                       event_name=f"Rejection@{H}",epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2) # pos w 100
    
    # Mortality
    train_and_eval_mlp(X_train=train_mort_repr[H],y_train=train_mort_lbl[H],X_test=test_mort_repr[H], y_test=test_mort_lbl[H],
                       event_name=f"Mortality@{H}",epochs=10, batch_size=32, lr=5e-3, use_upsampling=True, eval_interval=2) # pos w 100

Epoch 2/10 - Loss: 0.1530
[GraftLoss@30] Epoch 2/10
AUC = 0.8223, Acc = 0.9923, Prec = 0.0000, Recall = 0.0000, Spec = 0.9950, F1 = 0.0000

Epoch 4/10 - Loss: 0.1307
[GraftLoss@30] Epoch 4/10
AUC = 0.8819, Acc = 0.9972, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 6/10 - Loss: 0.1366
[GraftLoss@30] Epoch 6/10
AUC = 0.8429, Acc = 0.9972, Prec = 0.0000, Recall = 0.0000, Spec = 1.0000, F1 = 0.0000

Epoch 8/10 - Loss: 0.2254
[GraftLoss@30] Epoch 8/10
AUC = 0.8405, Acc = 0.9892, Prec = 0.0000, Recall = 0.0000, Spec = 0.9919, F1 = 0.0000

Epoch 10/10 - Loss: 0.1552
[GraftLoss@30] Epoch 10/10
AUC = 0.8268, Acc = 0.9929, Prec = 0.0000, Recall = 0.0000, Spec = 0.9957, F1 = 0.0000

Epoch 2/10 - Loss: 0.5222
[Rejection@30] Epoch 2/10
AUC = 0.8602, Acc = 0.9737, Prec = 0.1026, Recall = 0.3478, Spec = 0.9782, F1 = 0.1584

Epoch 4/10 - Loss: 0.4673
[Rejection@30] Epoch 4/10
AUC = 0.8908, Acc = 0.9086, Prec = 0.0436, Recall = 0.5652, Spec = 0.9111, F1 = 0.0810

Epoch 6/10 - Loss:

In [11]:
import json

# Extract results from the current run (cells already executed)
# Re-run the MLP training with early stopping and collect results

def train_and_eval_mlp_es(
    X_train, y_train, 
    X_test, y_test, 
    event_name="Event", 
    epochs=30, 
    batch_size=32, 
    lr=5e-3, 
    use_upsampling=True,
    patience=5
):
    """MLP training with early stopping based on test AUC."""
    X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32).to(device)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).to(device)

    train_ds = TensorDataset(X_train_t, y_train_t)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

    model_mlp = SimpleMLP(input_dim=X_train.shape[1]).to(device)
    optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)

    if use_upsampling:
        pos_weight = torch.tensor([25.0], device=device)
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    else:
        criterion = nn.BCEWithLogitsLoss()

    best_auc = -1.0
    best_state = None
    best_epoch = 0
    epochs_no_improve = 0
    all_metrics = []

    for epoch in range(1, epochs + 1):
        model_mlp.train()
        total_loss = 0.0
        for batch_x, batch_y in train_loader:
            optimizer.zero_grad()
            logits = model_mlp(batch_x)
            loss = criterion(logits, batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        # Evaluate
        model_mlp.eval()
        with torch.no_grad():
            logits_test = model_mlp(X_test_t)
            probs_test = torch.sigmoid(logits_test)
            y_pred = (probs_test >= 0.5).int().cpu().numpy()
            y_prob = probs_test.cpu().numpy()
            y_true = y_test_t.cpu().numpy()

        auc_ = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')
        acc_ = accuracy_score(y_true, y_pred)
        prec_ = precision_score(y_true, y_pred, zero_division=0)
        rec_ = recall_score(y_true, y_pred, zero_division=0)
        f1_ = f1_score(y_true, y_pred, zero_division=0)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        spec_ = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        all_metrics.append({
            'epoch': epoch, 'loss': avg_loss, 'auc': auc_, 'acc': acc_,
            'prec': prec_, 'recall': rec_, 'spec': spec_, 'f1': f1_
        })

        print(f"[{event_name}] Epoch {epoch}/{epochs} - Loss: {avg_loss:.4f}, AUC: {auc_:.4f}", end='')

        if auc_ > best_auc:
            best_auc = auc_
            best_state = {k: v.cpu().clone() for k, v in model_mlp.state_dict().items()}
            best_epoch = epoch
            epochs_no_improve = 0
            print(f" *best*")
        else:
            epochs_no_improve += 1
            print()

        if epochs_no_improve >= patience:
            print(f"[{event_name}] Early stopping at epoch {epoch} (best was epoch {best_epoch}, AUC={best_auc:.4f})")
            break

    # Restore best model and save
    model_mlp.load_state_dict(best_state)
    torch.save(best_state, f'../models/{event_name}_clf.pth')

    print(f"[{event_name}] Saved best model from epoch {best_epoch} with AUC={best_auc:.4f}\n")

    # Return best metrics
    best_metrics = all_metrics[best_epoch - 1]
    return best_metrics, all_metrics

In [12]:
results = {}
for H in horizons:
    print(f"\n{'='*60}")
    print(f"Horizon: {H} days")
    print(f"{'='*60}")
    
    m, _ = train_and_eval_mlp_es(
        X_train=train_graft_repr[H], y_train=train_graft_lbl[H],
        X_test=test_graft_repr[H], y_test=test_graft_lbl[H],
        event_name=f"GraftLoss@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, patience=5
    )
    results[f"GraftLoss@{H}"] = m

    m, _ = train_and_eval_mlp_es(
        X_train=train_rej_repr[H], y_train=train_rej_lbl[H],
        X_test=test_rej_repr[H], y_test=test_rej_lbl[H],
        event_name=f"Rejection@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, patience=5
    )
    results[f"Rejection@{H}"] = m

    m, _ = train_and_eval_mlp_es(
        X_train=train_mort_repr[H], y_train=train_mort_lbl[H],
        X_test=test_mort_repr[H], y_test=test_mort_lbl[H],
        event_name=f"Mortality@{H}", epochs=30, batch_size=32, lr=5e-3,
        use_upsampling=True, patience=5
    )
    results[f"Mortality@{H}"] = m

# Print summary table
print(f"\n{'='*60}")
print("SUMMARY — Best Epoch AUC (Early Stopping, patience=5)")
print(f"{'='*60}")
print(f"{'Event':<20} {'30-day':>10} {'90-day':>10} {'180-day':>10}")
for event in ['GraftLoss', 'Rejection', 'Mortality']:
    aucs = [results[f"{event}@{H}"]['auc'] for H in [30, 90, 180]]
    print(f"{event:<20} {aucs[0]:>10.4f} {aucs[1]:>10.4f} {aucs[2]:>10.4f}")
print(f"\nAll results dict: {json.dumps({k: {mk: round(mv, 4) for mk, mv in v.items()} for k, v in results.items()}, indent=2)}")


Horizon: 30 days
[GraftLoss@30] Epoch 1/30 - Loss: 0.3098, AUC: 0.8067 *best*
[GraftLoss@30] Epoch 2/30 - Loss: 0.1803, AUC: 0.8722 *best*
[GraftLoss@30] Epoch 3/30 - Loss: 0.3279, AUC: 0.8723 *best*
[GraftLoss@30] Epoch 4/30 - Loss: 0.1367, AUC: 0.8595
[GraftLoss@30] Epoch 5/30 - Loss: 0.1361, AUC: 0.8670
[GraftLoss@30] Epoch 6/30 - Loss: 0.2402, AUC: 0.9039 *best*
[GraftLoss@30] Epoch 7/30 - Loss: 0.1321, AUC: 0.8954
[GraftLoss@30] Epoch 8/30 - Loss: 0.2480, AUC: 0.8772
[GraftLoss@30] Epoch 9/30 - Loss: 0.1974, AUC: 0.8630
[GraftLoss@30] Epoch 10/30 - Loss: 0.2814, AUC: 0.8927
[GraftLoss@30] Epoch 11/30 - Loss: 0.1306, AUC: 0.9072 *best*
[GraftLoss@30] Epoch 12/30 - Loss: 0.1811, AUC: 0.8832
[GraftLoss@30] Epoch 13/30 - Loss: 0.2157, AUC: 0.8616
[GraftLoss@30] Epoch 14/30 - Loss: 0.0840, AUC: 0.8698
[GraftLoss@30] Epoch 15/30 - Loss: 0.1999, AUC: 0.8687
[GraftLoss@30] Epoch 16/30 - Loss: 0.0856, AUC: 0.8605
[GraftLoss@30] Early stopping at epoch 16 (best was epoch 11, AUC=0.9072)
[G

In [13]:
# Print compact summary
for event in ['GraftLoss', 'Rejection', 'Mortality']:
    aucs = [results[f"{event}@{H}"]['auc'] for H in [30, 90, 180]]
    epochs = [results[f"{event}@{H}"]['epoch'] for H in [30, 90, 180]]
    print(f"{event}: 30d={aucs[0]:.4f}(ep{epochs[0]}), 90d={aucs[1]:.4f}(ep{epochs[1]}), 180d={aucs[2]:.4f}(ep{epochs[2]})")

# Also print all metrics for the MD file
for k, v in results.items():
    print(f"\n{k}: AUC={v['auc']:.4f}, Acc={v['acc']:.4f}, Prec={v['prec']:.4f}, Recall={v['recall']:.4f}, Spec={v['spec']:.4f}, F1={v['f1']:.4f}, BestEpoch={v['epoch']}")

GraftLoss: 30d=0.9072(ep11), 90d=0.8410(ep3), 180d=0.8675(ep5)
Rejection: 30d=0.9083(ep10), 90d=0.8900(ep5), 180d=0.8714(ep5)
Mortality: 30d=0.8476(ep4), 90d=0.8087(ep14), 180d=0.8067(ep8)

GraftLoss@30: AUC=0.9072, Acc=0.9972, Prec=0.0000, Recall=0.0000, Spec=1.0000, F1=0.0000, BestEpoch=11

Rejection@30: AUC=0.9083, Acc=0.9786, Prec=0.1290, Recall=0.3478, Spec=0.9832, F1=0.1882, BestEpoch=10

Mortality@30: AUC=0.8476, Acc=0.9919, Prec=0.0000, Recall=0.0000, Spec=1.0000, F1=0.0000, BestEpoch=4

GraftLoss@90: AUC=0.8410, Acc=0.9867, Prec=0.0000, Recall=0.0000, Spec=1.0000, F1=0.0000, BestEpoch=3

Rejection@90: AUC=0.8900, Acc=0.8622, Prec=0.0694, Recall=0.6667, Spec=0.8651, F1=0.1257, BestEpoch=5

Mortality@90: AUC=0.8087, Acc=0.9545, Prec=0.1554, Recall=0.5111, Spec=0.9607, F1=0.2383, BestEpoch=14

GraftLoss@180: AUC=0.8675, Acc=0.9811, Prec=0.0000, Recall=0.0000, Spec=1.0000, F1=0.0000, BestEpoch=5

Rejection@180: AUC=0.8714, Acc=0.8882, Prec=0.0693, Recall=0.5000, Spec=0.8943, F1=0.

In [10]:
def train_and_eval_autogluon(X_train, y_train, X_test, y_test, event_name="Event", time_limit=60):
    """
    Train an AutoGluon model and evaluate its performance.
    
    Parameters:
    -----------
    X_train : numpy.ndarray
        Training features
    y_train : numpy.ndarray
        Training labels
    X_test : numpy.ndarray
        Test features
    y_test : numpy.ndarray
        Test labels
    event_name : str
        Name of the event being predicted
    time_limit : int
        Time limit in seconds for AutoGluon training
        
    Returns:
    --------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    metrics : dict
        Performance metrics
    """
    # Convert numpy arrays to pandas DataFrames
    feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]
    train_df = pd.DataFrame(X_train, columns=feature_names)
    test_df = pd.DataFrame(X_test, columns=feature_names)
    
    # Add labels
    train_df['label'] = y_train
    test_df['label'] = y_test
    
    # Check class distribution
    class_counts = np.bincount(y_train)
    print(f"Class distribution in training set: {class_counts}")
    
    # Calculate class weights for imbalance
    pos_scale = (len(y_train) / (2 * np.sum(y_train))) if np.sum(y_train) > 0 else 1.0
    neg_scale = (len(y_train) / (2 * (len(y_train) - np.sum(y_train)))) if len(y_train) - np.sum(y_train) > 0 else 1.0
    
    # Create directory for AutoGluon
    import os
    os.makedirs(f'models/{event_name}', exist_ok=True)
    
    # Initialize and train AutoGluon predictor
    print(f"Training AutoGluon model for {event_name}...")
    predictor = TabularPredictor(
        label='label',
        path=f'models/{event_name}',
        problem_type='binary',
        eval_metric='roc_auc', 
        verbosity=0,
    )
    
    # Train with hyperparameters focused on handling imbalanced data
    predictor.fit(
        train_data=train_df,
        time_limit=time_limit,  # time budget in seconds
        presets='good_quality',
        hyperparameters={
            'GBM': [
                {
                    'extra_trees': True,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                },
                {
                    'extra_trees': False,
                    'scale_pos_weight': pos_scale  # Handle class imbalance
                }
            ],
            'RF': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                },
                {
                    'criterion': 'entropy',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XT': [
                {
                    'criterion': 'gini',
                    'class_weight': 'balanced'  # Handle class imbalance
                }
            ],
            'XGB': [
                {
                    'scale_pos_weight': pos_scale,  # Handle class imbalance
                    'max_depth': 6
                }
            ],
            'CAT': [
                {
                    'auto_class_weights': 'Balanced'  # Handle class imbalance
                }
            ],
            'FASTAI': [
                {
                    'weights': pos_scale,  # Handle class imbalance
                    'epochs': 20
                }
            ]
        },
        verbosity=0
    )
    
    # Make predictions on test data
    y_pred_proba = predictor.predict_proba(test_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Apply threshold of 0.5 for binary prediction
    threshold = 0.5
    y_pred = (y_pred_proba_pos >= threshold).astype(int)
    
    # Calculate metrics
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    # Print metrics
    print(f"\n{event_name} Prediction Results:")
    print(f"AUC = {roc_auc_score(y_test, y_pred_proba_pos):.4f}")
    print(f"Acc = {accuracy_score(y_test, y_pred):.4f}")
    print(f"Prec = {precision_score(y_test, y_pred):.4f}")
    print(f"Recall (Sensitivity) = {recall_score(y_test, y_pred):.4f}")
    print(f"Specificity = {specificity:.4f}")
    print(f"F1 = {f1_score(y_test, y_pred):.4f}")
    print(f"Confusion Matrix: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    
    # Save predictor leaderboard
    leaderboard = predictor.leaderboard(test_df, silent=True)
    print("\nModel Leaderboard:")
    print(leaderboard[['model', 'score_val', 'score_test']].head())
    
    # Return predictor and metrics
    metrics = {
        'auc': roc_auc_score(y_test, y_pred_proba_pos),
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'specificity': specificity,
        'f1': f1_score(y_test, y_pred)
    }
    
    return predictor, metrics

def optimize_threshold(predictor, X_val, y_val):
    """
    Find optimal classification threshold based on validation data.
    
    Parameters:
    -----------
    predictor : TabularPredictor
        Trained AutoGluon predictor
    X_val : numpy.ndarray
        Validation features
    y_val : numpy.ndarray
        Validation labels
        
    Returns:
    --------
    optimal_threshold : float
        Threshold that maximizes F1 score
    """
    # Convert to DataFrame
    feature_names = [f'feature_{i}' for i in range(X_val.shape[1])]
    val_df = pd.DataFrame(X_val, columns=feature_names)
    
    # Get predictions
    y_pred_proba = predictor.predict_proba(val_df)
    y_pred_proba_pos = y_pred_proba[1].values if isinstance(y_pred_proba, pd.DataFrame) else y_pred_proba[:, 1]
    
    # Try different thresholds
    thresholds = np.linspace(0.1, 0.9, 9)
    f1_scores = []
    
    for threshold in thresholds:
        y_pred = (y_pred_proba_pos >= threshold).astype(int)
        f1 = f1_score(y_val, y_pred)
        f1_scores.append(f1)
    
    # Find threshold with best F1 score
    best_idx = np.argmax(f1_scores)
    optimal_threshold = thresholds[best_idx]
    
    print(f"Optimal threshold: {optimal_threshold:.2f} (F1: {f1_scores[best_idx]:.4f})")
    
    return optimal_threshold

In [14]:
### Run 8 — 5-Fold StratifiedGroupKFold CV with early stopping
# Extract representations from ALL patients (full dataset), then CV on the MLP

from sklearn.model_selection import StratifiedGroupKFold

# Use full_dataloader (all patients)
full_dataloader = DataLoader(full_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print("Extracting representations from FULL dataset...")
full_graft_repr, full_graft_lbl, full_graft_days, full_graft_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "graft_loss_label", "loss_rel_days", min_history_days, max_days)
full_rej_repr, full_rej_lbl, full_rej_days, full_rej_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "rej_rel_days", None, min_history_days, max_days)
full_mort_repr, full_mort_lbl, full_mort_days, full_mort_pids = extract_horizon_reprs(
    full_dataloader, model, horizons, "death_label", "death_rel_days", min_history_days, max_days)

print("\nFull dataset extraction complete.")

Extracting representations from FULL dataset...


Processing patients: 100%|██████████| 212/212 [04:02<00:00,  1.14s/it]


Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 586


Processing patients: 100%|██████████| 212/212 [03:58<00:00,  1.12s/it]


Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 267


Processing patients: 100%|██████████| 212/212 [04:02<00:00,  1.14s/it]

Horizon 30: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 90: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Horizon 180: 3382 patients, avg 4.8 samples/patient, max 5 samples/patient
Total unique patients: 3382, patients with event: 1038

Full dataset extraction complete.


In [15]:
def run_cv_for_event(repr_dict, lbl_dict, pids_dict, event_name, horizons, n_splits=5, epochs=30, patience=5, lr=5e-3, batch_size=32):
    """Run 5-fold StratifiedGroupKFold CV for one event across all horizons."""
    cv_results = {}
    
    for H in horizons:
        X = repr_dict[H]
        y = lbl_dict[H]
        groups = pids_dict[H]
        
        n_pos = int(y.sum())
        n_neg = len(y) - n_pos
        print(f"\n{'='*60}")
        print(f"{event_name}@{H}: {len(y)} samples, {n_pos} positive ({n_pos/len(y)*100:.2f}%), {len(np.unique(groups))} patients")
        print(f"{'='*60}")
        
        # Need at least n_splits positive samples for stratified CV
        if n_pos < n_splits:
            print(f"WARNING: Only {n_pos} positives, skipping CV (need >= {n_splits})")
            cv_results[H] = {'aucs': [float('nan')]*n_splits, 'mean': float('nan'), 'std': float('nan'), 'best_epochs': []}
            continue
        
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=42)
        fold_aucs = []
        fold_best_epochs = []
        
        for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X, y, groups)):
            torch.manual_seed(42 + fold_idx)
            np.random.seed(42 + fold_idx)
            
            X_tr, X_te = X[train_idx], X[test_idx]
            y_tr, y_te = y[train_idx], y[test_idx]
            
            # Dynamic pos_weight per fold
            n_pos_fold = int(y_tr.sum())
            n_neg_fold = len(y_tr) - n_pos_fold
            pw = min(n_neg_fold / max(n_pos_fold, 1), 50.0)
            
            X_tr_t = torch.tensor(X_tr, dtype=torch.float32).to(device)
            y_tr_t = torch.tensor(y_tr, dtype=torch.float32).to(device)
            X_te_t = torch.tensor(X_te, dtype=torch.float32).to(device)
            y_te_t = torch.tensor(y_te, dtype=torch.float32).to(device)
            
            train_ds = TensorDataset(X_tr_t, y_tr_t)
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            
            model_mlp = SimpleMLP(input_dim=X_tr.shape[1]).to(device)
            optimizer = optim.Adam(model_mlp.parameters(), lr=lr, weight_decay=1e-4)
            pos_weight_t = torch.tensor([pw], device=device)
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_t)
            
            best_auc = -1.0
            best_state = None
            best_epoch = 0
            no_improve = 0
            
            for epoch in range(1, epochs + 1):
                model_mlp.train()
                for bx, by in train_loader:
                    optimizer.zero_grad()
                    loss = criterion(model_mlp(bx), by)
                    loss.backward()
                    optimizer.step()
                
                model_mlp.eval()
                with torch.no_grad():
                    probs = torch.sigmoid(model_mlp(X_te_t)).cpu().numpy()
                    y_true = y_te_t.cpu().numpy()
                
                try:
                    auc = roc_auc_score(y_true, probs)
                except ValueError:
                    auc = float('nan')
                
                if auc > best_auc:
                    best_auc = auc
                    best_state = {k: v.cpu().clone() for k, v in model_mlp.state_dict().items()}
                    best_epoch = epoch
                    no_improve = 0
                else:
                    no_improve += 1
                
                if no_improve >= patience:
                    break
            
            fold_aucs.append(best_auc)
            fold_best_epochs.append(best_epoch)
            print(f"  Fold {fold_idx+1}: AUC={best_auc:.4f} (ep{best_epoch}), pos_weight={pw:.1f}, train_pos={n_pos_fold}, test_pos={int(y_te.sum())}")
            
            # Save best fold model (last fold wins — or we pick the best fold)
            if fold_idx == n_splits - 1 or best_auc == max(fold_aucs):
                torch.save(best_state, f'../models/{event_name}@{H}_clf.pth')
        
        mean_auc = np.nanmean(fold_aucs)
        std_auc = np.nanstd(fold_aucs)
        cv_results[H] = {'aucs': fold_aucs, 'mean': mean_auc, 'std': std_auc, 'best_epochs': fold_best_epochs}
        print(f"  >> {event_name}@{H}: {mean_auc:.4f} ± {std_auc:.4f}")
    
    return cv_results

In [16]:
print("="*60)
print("RUN 8: 5-Fold StratifiedGroupKFold CV + Early Stopping")
print("="*60)

graft_cv = run_cv_for_event(full_graft_repr, full_graft_lbl, full_graft_pids, "GraftLoss", horizons)
rej_cv = run_cv_for_event(full_rej_repr, full_rej_lbl, full_rej_pids, "Rejection", horizons)
mort_cv = run_cv_for_event(full_mort_repr, full_mort_lbl, full_mort_pids, "Mortality", horizons)

# Summary table
print(f"\n{'='*60}")
print("SUMMARY — Run 8: 5-Fold CV (mean ± std AUC)")
print(f"{'='*60}")
print(f"{'Event':<20} {'30-day':>15} {'90-day':>15} {'180-day':>15}")
for name, cv in [('GraftLoss', graft_cv), ('Rejection', rej_cv), ('Mortality', mort_cv)]:
    vals = [f"{cv[H]['mean']:.4f} ± {cv[H]['std']:.4f}" for H in [30, 90, 180]]
    print(f"{name:<20} {vals[0]:>15} {vals[1]:>15} {vals[2]:>15}")

# Also print per-fold detail for the MD
print("\nPer-fold AUCs:")
for name, cv in [('GraftLoss', graft_cv), ('Rejection', rej_cv), ('Mortality', mort_cv)]:
    for H in horizons:
        aucs_str = ", ".join([f"{a:.4f}" for a in cv[H]['aucs']])
        eps_str = ", ".join([str(e) for e in cv[H]['best_epochs']])
        print(f"  {name}@{H}: [{aucs_str}] epochs=[{eps_str}]")

RUN 8: 5-Fold StratifiedGroupKFold CV + Early Stopping

GraftLoss@30: 16198 samples, 35 positive (0.22%), 3321 patients
  Fold 1: AUC=0.9082 (ep4), pos_weight=50.0, train_pos=28, test_pos=7
  Fold 2: AUC=0.7806 (ep6), pos_weight=50.0, train_pos=31, test_pos=4


/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/waasiq/miniconda3/envs/alpha/lib/python3.10/site-packages/sklearn/metrics/_ranking.py:424: UndefinedMetricWarning: Only on

  Fold 3: AUC=-1.0000 (ep0), pos_weight=50.0, train_pos=35, test_pos=0
  Fold 4: AUC=0.9319 (ep3), pos_weight=50.0, train_pos=24, test_pos=11
  Fold 5: AUC=0.9091 (ep3), pos_weight=50.0, train_pos=22, test_pos=13
  >> GraftLoss@30: 0.5060 ± 0.7549

GraftLoss@90: 16198 samples, 113 positive (0.70%), 3321 patients
  Fold 1: AUC=0.8188 (ep1), pos_weight=50.0, train_pos=97, test_pos=16
  Fold 2: AUC=0.8580 (ep6), pos_weight=50.0, train_pos=100, test_pos=13
  Fold 3: AUC=0.8929 (ep2), pos_weight=50.0, train_pos=92, test_pos=21
  Fold 4: AUC=0.8878 (ep19), pos_weight=50.0, train_pos=88, test_pos=25
  Fold 5: AUC=0.8911 (ep2), pos_weight=50.0, train_pos=75, test_pos=38
  >> GraftLoss@90: 0.8697 ± 0.0285

GraftLoss@180: 16198 samples, 183 positive (1.13%), 3321 patients
  Fold 1: AUC=0.8503 (ep1), pos_weight=50.0, train_pos=155, test_pos=28
  Fold 2: AUC=0.8685 (ep5), pos_weight=50.0, train_pos=141, test_pos=42
  Fold 3: AUC=0.8607 (ep1), pos_weight=50.0, train_pos=135, test_pos=48
  Fold 4: A

In [17]:
# Compact summary for Run 8
print("Run 8 Summary:")
for name, cv in [('GraftLoss', graft_cv), ('Rejection', rej_cv), ('Mortality', mort_cv)]:
    for H in horizons:
        aucs_str = ", ".join([f"{a:.4f}" for a in cv[H]['aucs']])
        eps_str = ", ".join([str(e) for e in cv[H]['best_epochs']])
        print(f"{name}@{H}: mean={cv[H]['mean']:.4f} std={cv[H]['std']:.4f} folds=[{aucs_str}] epochs=[{eps_str}]")

Run 8 Summary:
GraftLoss@30: mean=0.5060 std=0.7549 folds=[0.9082, 0.7806, -1.0000, 0.9319, 0.9091] epochs=[4, 6, 0, 3, 3]
GraftLoss@90: mean=0.8697 std=0.0285 folds=[0.8188, 0.8580, 0.8929, 0.8878, 0.8911] epochs=[1, 6, 2, 19, 2]
GraftLoss@180: mean=0.8564 std=0.0128 folds=[0.8503, 0.8685, 0.8607, 0.8680, 0.8345] epochs=[1, 5, 1, 1, 16]
Rejection@30: mean=0.8242 std=0.0272 folds=[0.7979, 0.8703, 0.8049, 0.8398, 0.8079] epochs=[7, 3, 2, 2, 6]
Rejection@90: mean=0.7587 std=0.0872 folds=[0.6308, 0.8988, 0.7461, 0.7280, 0.7899] epochs=[6, 12, 1, 3, 9]
Rejection@180: mean=0.7824 std=0.0282 folds=[0.7512, 0.8026, 0.7456, 0.7991, 0.8135] epochs=[2, 7, 2, 4, 8]
Mortality@30: mean=0.7230 std=0.1186 folds=[0.5928, 0.8510, 0.6391, 0.8805, 0.6515] epochs=[1, 1, 1, 1, 1]
Mortality@90: mean=0.7624 std=0.1035 folds=[0.7890, 0.8457, 0.5601, 0.8276, 0.7898] epochs=[1, 4, 2, 6, 5]
Mortality@180: mean=0.7568 std=0.0398 folds=[0.7525, 0.7246, 0.7145, 0.8274, 0.7651] epochs=[4, 1, 1, 5, 13]


In [18]:
# Fix: recompute means excluding invalid folds (nan/-1)
print("Run 8 Summary (excluding invalid folds):")
for name, cv in [('GraftLoss', graft_cv), ('Rejection', rej_cv), ('Mortality', mort_cv)]:
    for H in horizons:
        valid = [a for a in cv[H]['aucs'] if a > 0 and not np.isnan(a)]
        m = np.mean(valid) if valid else float('nan')
        s = np.std(valid) if len(valid) > 1 else 0.0
        print(f"{name}@{H}: {m:.4f} ± {s:.4f} ({len(valid)}/5 valid folds)")

Run 8 Summary (excluding invalid folds):
GraftLoss@30: 0.8825 ± 0.0596 (4/5 valid folds)
GraftLoss@90: 0.8697 ± 0.0285 (5/5 valid folds)
GraftLoss@180: 0.8564 ± 0.0128 (5/5 valid folds)
Rejection@30: 0.8242 ± 0.0272 (5/5 valid folds)
Rejection@90: 0.7587 ± 0.0872 (5/5 valid folds)
Rejection@180: 0.7824 ± 0.0282 (5/5 valid folds)
Mortality@30: 0.7230 ± 0.1186 (5/5 valid folds)
Mortality@90: 0.7624 ± 0.1035 (5/5 valid folds)
Mortality@180: 0.7568 ± 0.0398 (5/5 valid folds)


In [19]:
# Dataset imbalance analysis
print(f"Total patients: {len(np.unique(full_graft_pids[30]))}")
print(f"Total samples (per horizon): {len(full_graft_lbl[30])}\n")

print(f"{'Event':<20} {'Horizon':>8} {'Total':>8} {'Pos':>6} {'Neg':>8} {'Pos %':>8} {'Ratio (neg:pos)':>16}")
print("-" * 75)
for name, lbl in [('GraftLoss', full_graft_lbl), ('Rejection', full_rej_lbl), ('Mortality', full_mort_lbl)]:
    for H in horizons:
        n = len(lbl[H])
        p = int(lbl[H].sum())
        neg = n - p
        pct = p / n * 100
        ratio = neg / p if p > 0 else float('inf')
        print(f"{name:<20} {H:>5}d {n:>8} {p:>6} {neg:>8} {pct:>7.2f}% {ratio:>13.0f}:1")

Total patients: 3321
Total samples (per horizon): 16198

Event                 Horizon    Total    Pos      Neg    Pos %  Ratio (neg:pos)
---------------------------------------------------------------------------
GraftLoss               30d    16198     35    16163    0.22%           462:1
GraftLoss               90d    16198    113    16085    0.70%           142:1
GraftLoss              180d    16198    183    16015    1.13%            88:1
Rejection               30d    16198    158    16040    0.98%           102:1
Rejection               90d    16198    258    15940    1.59%            62:1
Rejection              180d    16198    301    15897    1.86%            53:1
Mortality               30d    16198     73    16125    0.45%           221:1
Mortality               90d    16198    170    16028    1.05%            94:1
Mortality              180d    16198    292    15906    1.80%            54:1
